In [ ]:
from darts.models import TFTModel
from darts.dataprocessing.transformers import Scaler
import matplotlib.pyplot as plt
import pytorch_lightning as pl

# ==================================================
# 1. Single series, no covariates
# ==================================================
single_target = [all_targets[0]]  # just 1 series

train_single = [s.split_before(val_cutoff)[0] for s in single_target]

# ==================================================
# 2. Scale only the target
# ==================================================
scaler = Scaler()
train_single_scaled = scaler.fit_transform(train_single)

# ==================================================
# 3. Loss tracker
# ==================================================
class LossTracker(pl.Callback):
    def __init__(self):
        self.train_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("train_loss")
        if loss:
            self.train_losses.append(loss.item())
            print(f"Epoch {trainer.current_epoch+1} | train_loss: {loss.item():.6f}")

loss_tracker = LossTracker()

# ==================================================
# 4. Minimal model — no covariates, no static features
# ==================================================
overfit_model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=6,
    hidden_size=32,       # keep small
    lstm_layers=1,
    num_attention_heads=2,
    dropout=0.0,          # no dropout
    batch_size=8,         # small batch
    n_epochs=100,
    # NO categorical_embedding_sizes
    # NO add_encoders
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": [0],
        "precision": "32",         # full precision
        "enable_progress_bar": True,
        "callbacks": [loss_tracker],
    },
    random_state=42,
)

# ==================================================
# 5. Fit — no covariates at all
# ==================================================
overfit_model.fit(
    series=train_single_scaled,
    verbose=True,
)

# ==================================================
# 6. Plot
# ==================================================
plt.figure(figsize=(10, 4))
plt.plot(loss_tracker.train_losses, color="blue", label="Train Loss")
plt.xlabel("Epoch")
plt.ylabel("Quantile Loss")
plt.title("Minimal Overfit Test — No Covariates, Single Series")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"\nInitial loss: {loss_tracker.train_losses[0]:.6f}")
print(f"Final loss:   {loss_tracker.train_losses[-1]:.6f}")
print(f"Reduction:    {((loss_tracker.train_losses[0] - loss_tracker.train_losses[-1]) / loss_tracker.train_losses[0]) * 100:.1f}%")

In [ ]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    def __init__(self):
        self.losses = []
    
    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get('train_loss')
        if loss:
            self.losses.append(loss.item())

loss_logger = LossLogger()

model.fit(
    debug_ts,
    future_covariates=dummy_cov,
    verbose=True,
    trainer_kwargs={'callbacks': [loss_logger]}
)

# Plot
plt.plot(loss_logger.losses)
plt.xlabel('Epoch')
plt.ylabel('Train Loss')
plt.title('Overfitting Debug - Train Loss')
plt.show()